In [2]:
import sys
!{sys.executable} -m pip install pyspark

  Using cached pyspark-4.3.0-py2.py3-none-any.whl
  Using cached py4j-0.10.9.9-py2.py3-none-any.whl.metadata (1.3 kB)
Using cached py4j-0.10.9.9-py2.py3-none-any.whl (203 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("kedaibuku").getOrCreate()
LAKE = "../data/lake"

orders = spark.read.parquet(f"{LAKE}/silver/orders.parquet")
books = spark.read.parquet(f"{LAKE}/silver/books.parquet")

sales = (orders.join(books.select("book_id", "category", "price_myr"), "book_id")
         .withColumn("revenue_myr", F.round(F.col("quantity") * F.col("price_myr"), 2))
         .withColumn("order_month", F.date_format("order_date", "yyyy-MM")))

# Write one folder per month
sales.write.mode("overwrite").partitionBy("order_month").parquet(f"{LAKE}/gold/sales_by_month")

june = spark.read.parquet(f"{LAKE}/gold/sales_by_month").filter(F.col("order_month") == "2026-06")
print(june.count())
june.explain()

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.